<a href="https://colab.research.google.com/github/CharithManaujayaMUTEC/ChemGenAI/blob/main/ChemGenAI_Day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch pandas numpy matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.6/530.6 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [3]:
df = pd.DataFrame({
    "smiles": [
        "CCO",
        "CCN",
        "CCC",
        "CC(=O)O",
        "C1=CC=CC=C1",
        "CCCl",
        "CCBr",
        "CCOC(=O)C",
        "CN",
        "CO"
    ]
})

df

,smiles
0,CCO
1,CCN
2,CCC
3,CC(=O)O
4,C1=CC=CC=C1
5,CCCl
6,CCBr
7,CCOC(=O)C
8,CN
9,CO


In [4]:
smiles_list = df["smiles"].tolist()

chars = sorted(list(set("".join(smiles_list))))

char_to_idx = {c:i+1 for i,c in enumerate(chars)}
idx_to_char = {i:c for c,i in char_to_idx.items()}

print(char_to_idx)

{'(': 1, ')': 2, '1': 3, '=': 4, 'B': 5, 'C': 6, 'N': 7, 'O': 8, 'l': 9, 'r': 10}


In [5]:
max_len = max(len(s) for s in smiles_list)

encoded = []

for s in smiles_list:
    seq = [char_to_idx[ch] for ch in s]
    seq += [0]*(max_len - len(seq))
    encoded.append(seq)

X = np.array(encoded)
print(X.shape)
print(X)

(10, 11)
[[ 6  6  8  0  0  0  0  0  0  0  0]
 [ 6  6  7  0  0  0  0  0  0  0  0]
 [ 6  6  6  0  0  0  0  0  0  0  0]
 [ 6  6  1  4  8  2  8  0  0  0  0]
 [ 6  3  4  6  6  4  6  6  4  6  3]
 [ 6  6  6  9  0  0  0  0  0  0  0]
 [ 6  6  5 10  0  0  0  0  0  0  0]
 [ 6  6  8  6  1  4  8  2  6  0  0]
 [ 6  7  0  0  0  0  0  0  0  0  0]
 [ 6  8  0  0  0  0  0  0  0  0  0]]


In [6]:
X_tensor = torch.tensor(X, dtype=torch.float32)

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

In [8]:
input_dim = X.shape[1]      # sequence length
hidden_dim = 64
latent_dim = 16

In [9]:
class VAE(nn.Module):
    def __init__(self):
        super(VAE, self).__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return self.fc3(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        output = self.decode(z)
        return output, mu, logvar

In [10]:
model = VAE()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
def loss_function(recon_x, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss

In [12]:
epochs = 300

for epoch in range(epochs):
    optimizer.zero_grad()

    recon, mu, logvar = model(X_tensor)
    loss = loss_function(recon, X_tensor, mu, logvar)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.2f}")

Epoch 0, Loss: 1871.40
Epoch 50, Loss: 821.40
Epoch 100, Loss: 428.73
Epoch 150, Loss: 338.58
Epoch 200, Loss: 325.59
Epoch 250, Loss: 209.26


In [13]:
model.eval()

with torch.no_grad():
    recon, _, _ = model(X_tensor)

print(recon[0])

tensor([ 5.4810,  5.5597,  6.5523,  0.9928, -0.8366,  0.1484,  0.1171,  0.3946,
         0.8543,  0.2422, -0.0898])


In [14]:
model.eval()

VAE(
  (fc1): Linear(in_features=11, out_features=64, bias=True)
  (fc_mu): Linear(in_features=64, out_features=16, bias=True)
  (fc_logvar): Linear(in_features=64, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=11, bias=True)
)

In [15]:
z = torch.randn(1, latent_dim)
decoded = model.decode(z)
decoded

tensor([[ 3.4680,  3.6005,  1.8852, -0.2993,  1.1741,  0.2401,  0.7461, -0.2111,
         -0.1479, -0.4215, -0.2577]], grad_fn=<AddmmBackward0>)

In [16]:
generated = decoded.detach().numpy()[0]
generated

array([ 3.468012  ,  3.6004965 ,  1.885239  , -0.29930955,  1.1741153 ,
        0.24006455,  0.74612004, -0.21114129, -0.1478829 , -0.4215415 ,
       -0.25765395], dtype=float32)

In [17]:
tokens = np.round(generated).astype(int)
tokens

array([3, 4, 2, 0, 1, 0, 1, 0, 0, 0, 0])

In [18]:
tokens = np.clip(tokens, 1, len(idx_to_char))
tokens

array([3, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1])

In [19]:
smiles = "".join([idx_to_char[t] for t in tokens])
print("Generated Molecule:", smiles)

Generated Molecule: 1=)((((((((


In [20]:
for i in range(10):
    z = torch.randn(1, latent_dim)
    decoded = model.decode(z).detach().numpy()[0]

    tokens = np.round(decoded).astype(int)
    tokens = np.clip(tokens, 1, len(idx_to_char))

    smiles = "".join([idx_to_char[t] for t in tokens])
    print(i+1, smiles)

1 11)()()((((
2 )(((1)11)1)
3 ))(((((((((
4 ))((1(1((((
5 ===1(((()((
6 1))((((((((
7 111)(((((((
8 =1==(()()((
9 (((((((((((
10 BB=)(((((((


In [21]:
!pip install datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 150.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.0 MB/s eta 0:00:00


In [22]:
from datasets import load_dataset

dataset = load_dataset("sagawa/ZINC-canonicalized")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

data/train-00000-of-00003-1dd8e62fc25564(…):   0%|          | 0.00/175M [00:00<?, ?B/s]

data/train-00001-of-00003-f5a496221de849(…):   0%|          | 0.00/175M [00:00<?, ?B/s]

data/train-00002-of-00003-1bb1afcf507d86(…):   0%|          | 0.00/175M [00:00<?, ?B/s]

data/validation-00000-of-00001-53255e680(…):   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20693269 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2299253 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['smiles'],
        num_rows: 20693269
    })
    validation: Dataset({
        features: ['smiles'],
        num_rows: 2299253
    })
})


In [23]:
train_data = dataset["train"].select(range(50000))
print(train_data)

Dataset({
    features: ['smiles'],
    num_rows: 50000
})


In [24]:
train_data[:5]

{'smiles': ['O=C1NCCN1[C@@H]1CCC[NH+](Cc2cccc(O)c2)C1',
  'CC(=O)N1c2ccc(S(=O)(=O)N3CCCC3)cc2C[C@@H]1C(=O)NCCN1CCCCCC1',
  'Cc1nc(-c2ccc(NC(=O)[C@H]3C[C@H]4CC[C@@H]3O4)cc2)cs1',
  'COc1ccc(C(=O)[C@H](C)Sc2nc(-c3cccs3)n[n-]2)cc1OC',
  'CCOC(=O)c1sc(NC(=O)CCCS(=O)(=O)c2ccc(F)cc2)nc1-c1ccccc1']}

In [25]:
df = train_data.to_pandas()
df.head()

,smiles
0,O=C1NCCN1[C@@H]1CCC[NH+](Cc2cccc(O)c2)C1
1,CC(=O)N1c2ccc(S(=O)(=O)N3CCCC3)cc2C[C@@H]1C(=O...
2,Cc1nc(-c2ccc(NC(=O)[C@H]3C[C@H]4CC[C@@H]3O4)cc...
3,COc1ccc(C(=O)[C@H](C)Sc2nc(-c3cccs3)n[n-]2)cc1OC
4,CCOC(=O)c1sc(NC(=O)CCCS(=O)(=O)c2ccc(F)cc2)nc1...


In [26]:
df["length"] = df["smiles"].apply(len)
df["length"].describe()

,length
count,50000.00000
mean,44.79232
std,9.28340
min,8.00000
25%,39.00000
50%,44.00000
75%,50.00000
max,134.00000
